# DPCK–MoE: OLMoE prefill router-trace capture (Colab T4)

**Status:** reproducibility scaffold; not executed or validated in this authoring session.

This notebook captures **prefill-only** routing traces. It records the exact expert indices and scores returned by each OLMoE router gate and checks them against a fresh Top-$K$ reconstruction from the model's returned router logits. Any mismatch aborts export.

It does **not** measure multi-GPU communication, all-to-all latency, decode-time routing, or serving throughput. Those require a separate expert-parallel testbed. Do not use the included smoke prompts as experimental evidence.

## Before running
1. In Colab, select **Runtime → Change runtime type → T4 GPU**.
2. Upload `trace_capture.py` and a `prompts.csv` file with columns `session_id,family,prompt`.
3. Use prompts that you are authorized to process and disclose dataset name, version, licensing, sampling and prompt template in your experiment record.
4. Start with the bundled `prompts_smoke.csv` only to test the pipeline. Replace it with the locked research prompt set for actual experiments.
5. The first model download resolves a repository commit. Record it from `manifest.json` and pin that exact SHA for subsequent runs.


In [ ]:
# Runtime and hardware verification. This deliberately fails closed on non-T4 GPUs.
import os, sys, json, platform, subprocess, textwrap
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA runtime found. Select a GPU runtime in Colab.")
DEVICE_NAME = torch.cuda.get_device_name(0)
print("GPU:", DEVICE_NAME)
print("Properties:", torch.cuda.get_device_properties(0))
if "T4" not in DEVICE_NAME.upper():
    raise RuntimeError(f"This notebook is configured for a Tesla T4, but detected: {DEVICE_NAME}")


## Install the capture dependencies

The dependency range is intentionally bounded but not frozen to an untested patch-level lockfile. The actual installed versions are written to the run manifest. After a successful pilot, freeze those exact versions and the resolved model commit for the main experiment.

In [ ]:
%pip -q install "transformers>=4.45.0,<6" "accelerate>=0.34,<2" "bitsandbytes>=0.45,<1" "safetensors>=0.4.5" pandas


## Upload capture script and prompt manifest

`prompts.csv` must contain `family` and `prompt`. `session_id` is strongly recommended and should identify an independent prompt/session. Keep the raw prompt manifest in a controlled location; the exported router trace contains only a SHA-256 prompt hash, not prompt text.

In [ ]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()  # Select trace_capture.py and prompts.csv (or prompts_smoke.csv).
if "trace_capture.py" not in uploaded:
    raise RuntimeError("Upload the companion trace_capture.py file.")
Path("trace_capture.py").write_bytes(uploaded["trace_capture.py"])
if "prompts.csv" in uploaded:
    PROMPTS_FILE = "prompts.csv"
elif "prompts_smoke.csv" in uploaded:
    PROMPTS_FILE = "prompts_smoke.csv"
    print("WARNING: bundled prompts are smoke-test examples, not research data.")
    Path(PROMPTS_FILE).write_bytes(uploaded[PROMPTS_FILE])
else:
    raise RuntimeError("Upload prompts.csv or prompts_smoke.csv.")
print("Prompt input:", PROMPTS_FILE)


## Capture routing traces

The default loads int8 weights to reduce T4 memory pressure. Quantization can change router outputs. The run manifest records the precision, and a separate precision-sensitivity experiment is mandatory before making precision-invariance claims.

Set `MODEL_REVISION` to an exact Hugging Face commit SHA after the initial resolution run. For the very first bootstrap only, `None` is permitted and the resolved SHA is written to the manifest.

In [ ]:
import os, subprocess

MODEL_REVISION = os.environ.get("MODEL_REVISION") or None  # pin to resolved SHA on repeat runs
cmd = [
    sys.executable, "trace_capture.py",
    "--prompts", PROMPTS_FILE,
    "--output-dir", "dpck_capture",
    "--max-length", "256",
    "--window-tokens", "64",
    "--load-8bit",
    "--require-t4",
]
if MODEL_REVISION:
    cmd += ["--revision", MODEL_REVISION]
if PROMPTS_FILE == "prompts_smoke.csv":
    cmd += ["--smoke-test"]
print("Command:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
# Inspect the run manifest and verify the exported JSONL structure.
from pathlib import Path
import json

manifest = json.loads(Path("dpck_capture/manifest.json").read_text())
print(json.dumps({k: manifest[k] for k in [
    "model_id", "model_revision_resolved", "model_config", "capture_scope",
    "precision", "prompt_count", "routed_tokens_per_layer", "trace_records",
    "smoke_test_only", "trace_sha256"
]}, indent=2))

trace_path = Path("dpck_capture/trace.jsonl")
first = json.loads(trace_path.open().readline())
required = {"run_id", "model_id", "prompt_hash", "session_id", "window_id",
            "token_position", "phase", "layer_id", "source_rank", "expert_ids",
            "routing_weights", "precision", "quantization"}
missing = required - set(first)
assert not missing, f"Missing required trace fields: {sorted(missing)}"
assert first["phase"] == "prefill"
assert len(first["expert_ids"]) == manifest["model_config"]["num_experts_per_tok"]
print("Schema smoke check: PASS")
print("First record (prompt text is not present):", json.dumps(first, indent=2))


In [ ]:
# Archive immutable trace + manifest + router fidelity report and download.
import zipfile
from google.colab import files

archive = Path("dpck_capture_artifact.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in Path("dpck_capture").glob("*"):
        zf.write(path, arcname=f"dpck_capture/{path.name}")
files.download(str(archive))


## Acceptance checks before using the trace in a paper

- [ ] Detected device is Tesla T4 and its properties are in the manifest.
- [ ] Exact model revision SHA is recorded and pinned for later runs.
- [ ] `router_fidelity.json` reports exact index agreement for every layer and every captured token.
- [ ] Trace contains all expected MoE layers and exactly Top-8 expert IDs per event for OLMoE-1B-7B-0125.
- [ ] Dataset source, revision, prompt selection, license, and independent session IDs are documented.
- [ ] Smoke-test prompts are excluded from research analyses.
- [ ] Quantization sensitivity is tested; no route-invariance assumption is made.
- [ ] The trace is treated as **prefill-only**; no decode or distributed-runtime claim is inferred.
- [ ] The SHA-256 trace hash and run manifest are archived with every result.

The first-stage single-GPU trace uses `source_rank=-1` as an explicit **unknown** sentinel. It must not be interpreted as a physical expert-parallel source rank. Simulated topology experiments must separately declare a synthetic ingress mapping and be reported as simulations.